## Langchain prompt templates
differeance between prompt template & chatprompt template

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAI
from langchain_ollama import ChatOllama

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)


---
## 1) PromptTemplate vs ChatPromptTemplate

LangChain has **2 types** of prompt templates:

| | PromptTemplate | ChatPromptTemplate |
|---|---|---|
| Type | Simple string | Conversation style |
| Has roles? | ❌ No | ✅ Yes (system, human) |
| Use with | Old LLMs | Modern Chat models |

### What are roles in ChatPromptTemplate?
- **system** → Tells the AI how to behave (like giving instructions)
- **human** → The user's actual question/input
- **ai** → AI's previous response (used in conversations)

In [6]:
prompt1=PromptTemplate(
    input_variables=['sentence1','sentence2'],
    template='Merge the two sentence into one . sentence1:{sentence1},sentence2:{sentence2}',
)
print("prompt template output")
print(prompt1.format(sentence1="my name is yasin",sentence2="i am doing coding"),"\n")

prompt template output
Merge the two sentence into one . sentence1:my name is yasin,sentence2:i am doing coding 



In [9]:
prompt2=ChatPromptTemplate.from_template(
    template="merge the two sentence into one . sentence1:{sentence1},sentence2:{sentence2}"
)
print("ChatPromptTemplate.from_template")
print(prompt2.format(sentence1="i am yasin",sentence2="male"),"\n")

ChatPromptTemplate.from_template
Human: merge the two sentence into one . sentence1:i am yasin,sentence2:male 



In [11]:
prompt3=ChatPromptTemplate.from_messages([
    ("system","always gives the correct answer"),
    ("human","Merge the two sentence into one. sentence1:{sentence1},sentence 2:{sentence2}")
])

print("--- ChatPromptTemplate.from_messages output ---")
print(prompt3.format(sentence1="What is the capital of France?", sentence2="Paris"), "\n")


--- ChatPromptTemplate.from_messages output ---
System: always gives the correct answer
Human: Merge the two sentence into one. sentence1:What is the capital of France?,sentence 2:Paris 



---
# Part 2: Model Parameters

When creating an LLM, you can control its behaviour using parameters:

| Parameter | What it does | Example |
|---|---|---|
| `temperature` | Controls creativity | `0` = focused, `1` = creative |
| `max_tokens` | Limits response length | `30` = max 30 words |
| `stop` | Stops generating at a word | `stop="India"` |
| `timeout` | Max wait time in seconds | `timeout=30` |

In [13]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,         # 0 = focused/factual answers (no creativity)
    max_tokens=30,         # response will be max 30 tokens (words)
    timeout=30             # if no response in 30 seconds → error
)

res = llm.invoke("Where is India located?")
print(res.content)

India is a country located in South Asia. It is situated on the Indian subcontinent, which is a large peninsula that extends into the Indian Ocean.


*Temperature*

In [ ]:

# Focused answer (temperature=0)
llm_focused = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,         
    max_tokens=30,    
    timeout=30      
)
# Focused answer (temperature=1)
llm_creative = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=1,         
    max_tokens=30,    
    timeout=30             
)

print("Focused (temp=0):", llm_focused.invoke("Write a word that means happy").content)
print("Creative (temp=1):", llm_creative.invoke("Write a word that means happy").content)

Focused (temp=0): The word is "joyful".
Creative (temp=1): The word "happy" itself is one possible answer. Some other options could be:

- Joyful
- Jubilant
- Elated



---
# Part 3: Text Completion vs Chat Models

There are **2 types** of LLM models in LangChain:

| | Text Completion (Old) | Chat Model (Modern ✅) |
|---|---|---|
| Class | `OpenAI()` / `GoogleGenerativeAI()` | `ChatOpenAI()` / `ChatGoogleGenerativeAI()` |
| Input | Plain string | Messages with roles |
| Output | Plain string | AIMessage object |
| Get text | `res` directly | `res.content` |
| Use today? | ❌ Old way | ✅ Always use this |

### Simple analogy:
- **Text Completion** = Like sending an SMS (plain text, no context)
- **Chat Model** = Like WhatsApp chat (has roles, context, conversation)

In [ ]:
# Text Completion Model (OLD way) ❌

# - Uses GoogleGenerativeAI (not Chat version)
# - Returns plain string directly
# - No roles, no system message

text_llm = GoogleGenerativeAI(
    temperature=0,
    model="gemini-1.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

res = text_llm.invoke("Where is India located?")

print(type(res))   # plain string
print(res)         # no .content needed!

In [ ]:
# Chat Model (MODERN way) ✅

# - Uses ChatGoogleGenerativeAI
# - Returns AIMessage object
# - Supports roles (system, human, ai)
# - Always use this for modern LangChain apps!

chat_llm = ChatGoogleGenerativeAI(
    temperature=0,
    model="gemini-1.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

res = chat_llm.invoke("Where is India located?")

print(type(res))       # AIMessage object
print(res.content)     # use .content to get text

In [ ]:
# Full example: ChatPromptTemplate + Chat Model
# This is the MODERN and RECOMMENDED way!

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always answer in one sentence."),
    ("human", "{input}")
])

chain = prompt | chat_llm

res = chain.invoke({"input": "Where is India located?"})
print(res.content)

---
## 📌 Part 2 Summary

| Concept | Key Point |
|---|---|
| `PromptTemplate` | Simple string template, no roles |
| `ChatPromptTemplate.from_template` | Simple chat template, auto HumanMessage |
| `ChatPromptTemplate.from_messages` | Full control with system + human roles ✅ |
| `temperature=0` | Focused, factual answers |
| `temperature=1` | Creative, random answers |
| `max_tokens` | Limit the response length |
| `GoogleGenerativeAI` | Text completion (old) → returns string |
| `ChatGoogleGenerativeAI` | Chat model (modern) → returns AIMessage ✅ |

### ✅ Best practice for modern LangChain:
```python
# Always use these two together!
prompt = ChatPromptTemplate.from_messages([...])
llm    = ChatGoogleGenerativeAI(...)
chain  = prompt | llm
```